# TOOLS

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.Tools are pairings of:

1. A schema , including the name of the tool, a description , and/or argument definitions (often a JASON schema).
2. A function or coroutine to execute.

In [34]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model= init_chat_model("llama-3.3-70b-versatile",
                       model_provider= "groq"
                       )
response=model.invoke("shinchan is better than doraemon.")
response


AIMessage(content="That's an interesting opinion. Both Shinchan and Doraemon are popular Japanese manga and anime series, but they cater to different age groups and have distinct themes.\n\nShinchan is known for its comedic and satirical take on everyday life, following the misadventures of a 5-year-old boy named Shinnosuke Nohara. The series is famous for its crude humor, witty dialogue, and relatable portrayal of childhood.\n\nOn the other hand, Doraemon is a more classic and iconic series that focuses on science fiction and adventure. It follows the story of a robotic cat named Doraemon who travels back in time to help his friend Noby, a young boy who often gets into trouble.\n\nWhile both series have their unique charm, it ultimately comes down to personal preference. Some people might enjoy Shinchan's humor and satire, while others might prefer Doraemon's imaginative and futuristic storylines.\n\nWhat do you like about Shinchan that makes you think it's better than Doraemon? Is it

In [36]:
from langchain.tools import tool
@tool # @-> decorator
def get_weather(location:str)->str:
    """Get the weather for the requested location.
    The returned value is the weather result and should be used
    to answer the user's question"""
    return f"its sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [38]:
response = model_with_tools.invoke("what's the weather like in Buston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")
    
tool_call = response.tool_calls[0]
result = get_weather.invoke(tool_call["args"])
print(result)


content='' additional_kwargs={'tool_calls': [{'id': '1yst725jh', 'function': {'arguments': '{"location":"Buston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 262, 'total_tokens': 278, 'completion_time': 0.032259124, 'completion_tokens_details': None, 'prompt_time': 0.01350495, 'prompt_tokens_details': None, 'queue_time': 0.161429977, 'total_time': 0.045764074}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f328b-f29f-7df1-8bc4-893e4563d6bc-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Buston'}, 'id': '1yst725jh', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 262, 'output_tokens': 16, 'total_tokens': 278}
Tool:get_weather
Args:{'location': 'Buston'}
its sunny in Buston


### Tool Execution Loops

In [39]:
from langchain_core.messages import ToolMessage
# Step 1: model generates tool call
messages = [
    {
        "role": "user",
        "content": "What's the weather in Boston?"
    }
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# Step 2: execute tools
for tool_call in ai_msg.tool_calls:
    result = get_weather.invoke(tool_call["args"])
    tool_result = ToolMessage(
        content=result,
        tool_call_id=tool_call["id"]
    )
    messages.append(tool_result)
# Step 3: send conversation back to model
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The current weather in Boston is sunny.


In [40]:
for msg in messages:
    print(type(msg).__name__)
    print(msg)
    print("-"*50)

dict
{'role': 'user', 'content': "What's the weather in Boston?"}
--------------------------------------------------
AIMessage
content='' additional_kwargs={'tool_calls': [{'id': 'ysacqb0c3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 260, 'total_tokens': 274, 'completion_time': 0.030841997, 'completion_tokens_details': None, 'prompt_time': 0.013908791, 'prompt_tokens_details': None, 'queue_time': 0.053165708, 'total_time': 0.044750788}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f328c-21f5-7632-b484-881b637db4b4-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'ysacqb0c3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 260, 'output_tokens': 14, 't